In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Step 1: Get app metadata (rating, installs, description...) for CBE, BOA, Dashen banks

# Map clean display names to their respective Play Store App IDs

banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Iteratively fetch and display app metadata
for bank_name, app_id in banks.items():
    try:
    # Fetch data directly inside the loop using the current app_id
        app_info = app(app_id, lang='en', country='et')     # Language: English, Country: Ethiopia
    
        print("=" * 50)
        print(f"{bank_name} App Info")
        print("=" * 50)
        print(f"App Title    : {app_info['title']}")
        print(f"Current Score: {app_info['score']}")
        print(f"Total Ratings: {app_info['ratings']:,}")
        print(f"Total Reviews: {app_info['reviews']:,}")
        print(f"Installs     : {app_info['installs']}\n")

    except Exception as e:
        # If an ID fails (like a 404), catch it here and keep going
        print("=" * 50)
        print(f"⚠️ Error loading {bank_name}")
        print("=" * 50)
        print(f"Could not retrieve App ID: '{app_id}'")
        print(f"Details: {e}\n")

Commercial Bank of Ethiopia App Info
App Title    : Commercial Bank of Ethiopia
Current Score: 4.289424
Total Ratings: 48,384
Total Reviews: 9,316
Installs     : 5,000,000+

Bank of Abyssinia App Info
App Title    : BoA Mobile
Current Score: 4.3876925
Total Ratings: 9,234
Total Reviews: 1,461
Installs     : 1,000,000+

Dashen Bank App Info
App Title    : Dashen Bank
Current Score: 4.2579503
Total Ratings: 5,644
Total Reviews: 1,023
Installs     : 1,000,000+



In [7]:
# Step 2: Scrape reviews

# Dictionary to map clean display names to their App IDs
banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Master dictionary to store the actual raw reviews lists for each bank
all_bank_reviews = {}

# 1. Iterate through each bank to scrape reviews
for bank_name, app_id in banks.items():  
    try:
        # Scrape reviews
        result, continuation_token = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,       # Most recent first
            count=500,              # Target count
            filter_score_with=None  # All star ratings
        )
        
        # Store the list of reviews using the bank's name as the key
        all_bank_reviews[bank_name] = result
        print(f"✅ Success: Collected {len(result)} raw reviews for {bank_name}\n")
        
    except Exception as e:
        # **Safety Net:** If one bank fails (e.g., a temporary network glitch or an ID change), it catches errors and keep the loop running
        print(f"❌ Error scraping {bank_name} due to error: {e}\n")

# --- FINAL VALIDATION CHECK ---
print("=" * 50)
print("FINAL COLLECTION CHECK SUMMARY")
print("=" * 50)
for bank, status in all_bank_reviews.items():
    print(f"{bank:<30} : {len(status)} Raw Reviews")
print("=" * 50)

# 2. Inspect a single raw review from each bank
print("=" * 60)
print("INSPECTING A SAMPLE REVIEW FOR EACH BANK")
print("=" * 60)

# Iterate through all banks in the collected data
for bank_name, reviews_list in all_bank_reviews.items():
    print(f"\nTarget Bank: {bank_name}")
    print("-" * 50)
    
    if reviews_list and len(reviews_list) > 0:
        # Grab the very first review dictionary from the current bank's list
        sample_review = reviews_list[0]
        
        # Display all the available keys safely
        print(f"Keys available in this review: {list(sample_review.keys())}\n")
        print("First raw review data details:")
        
        for key, value in sample_review.items():
            print(f" {key:<20}: {value}")
            
    else:
        print(f"No sample data available for {bank_name}. Check your collection step.")
    print("-" * 50)

✅ Success: Collected 500 raw reviews for Commercial Bank of Ethiopia

✅ Success: Collected 500 raw reviews for Bank of Abyssinia

✅ Success: Collected 500 raw reviews for Dashen Bank

FINAL COLLECTION CHECK SUMMARY
Commercial Bank of Ethiopia    : 500 Raw Reviews
Bank of Abyssinia              : 500 Raw Reviews
Dashen Bank                    : 500 Raw Reviews
INSPECTING A SAMPLE REVIEW FOR EACH BANK

Target Bank: Commercial Bank of Ethiopia
--------------------------------------------------
Keys available in this review: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review data details:
 reviewId            : ba0c5d66-8085-4bff-908b-f553c7b14ff5
 userName            : Yalew Mamed
 userImage           : https://play-lh.googleusercontent.com/a/ACg8ocIJXtC-Q6pj9HbzXLNKgIBuWYQD_rm08H-GXwurQhvLOC88Yg=mo
 content             : It's not allowing me to transfer money.
 score        

In [14]:
# Step 3: Extract only the columns needed from all banks
combined_raw_data = []

for bank_name, reviews_list in all_bank_reviews.items():
    print(f"Processing and extracting columns for: {bank_name}...")
    
    for r in reviews_list:
        combined_raw_data.append({
            'review_id': r.get('reviewId', ''),
            'review'   : r.get('content', ''),
            'rating'   : r.get('score', None),
            'date'     : r.get('at', None),
            'bank'     : bank_name,          # Dynamically tags the correct bank name
            'source'   : 'Google Play'
        })

# Build a single master DataFrame
df_raw = pd.DataFrame(combined_raw_data)

print("\n" + "=" * 50)
print(f"Extraction Complete!")
print(f"Final Combined DataFrame Shape: {df_raw.shape}")
print("=" * 50)

# Display a breakdown of reviews collected per bank
print("\nReviews per bank in DataFrame:")
print(df_raw['bank'].value_counts())

# Randomly select and display 5 reviews in a table format
df_raw.sample(5)

Processing and extracting columns for: Commercial Bank of Ethiopia...
Processing and extracting columns for: Bank of Abyssinia...
Processing and extracting columns for: Dashen Bank...

Extraction Complete!
Final Combined DataFrame Shape: (1500, 6)

Reviews per bank in DataFrame:
bank
Commercial Bank of Ethiopia    500
Bank of Abyssinia              500
Dashen Bank                    500
Name: count, dtype: int64


,review_id,review,rating,date,bank,source
576,497be43a-22d6-41de-b1b8-25b6a9828edb,its the the bank of bank,5,2026-03-25 10:33:59,Bank of Abyssinia,Google Play
1393,befc23dd-22be-4039-ab16-311ebf096360,It takes gazillion years to open 😶,3,2025-10-02 10:54:44,Dashen Bank,Google Play
27,f9d96cc8-f98d-4cd4-9d0c-7ce9af07256d,posetive,5,2026-05-10 07:13:20,Commercial Bank of Ethiopia,Google Play
668,e1556dbc-4ca8-4d4f-a9e5-f2bf4a8559a5,"balance sync is working after 24 hours, freque...",2,2026-02-23 08:12:09,Bank of Abyssinia,Google Play
431,c83ece70-b75a-4917-99d7-29adfe39cad1,proud of CBE. CBE is my everyday day choice an...,5,2026-03-15 15:55:23,Commercial Bank of Ethiopia,Google Play


In [ ]:
# A) Exploring the merged raw data

# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)


# Rating distribution — what do users think?
print("\nRating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")


# What does the date column look like right now?
print("\nSample date values (raw):")
print(df_raw['date'].sample(5).to_string())   #Randomly select date to notice differences

print(f"\nDate dtype: {df_raw['date'].dtype}")

Total reviews collected: 1500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object

Rating distribution:
  5 stars:  941  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:  113  ██████████████████████
  3 stars:   79  ███████████████
  2 stars:   48  █████████
  1 stars:  319  ███████████████████████████████████████████████████████████████

Sample date values (raw):
480   2026-03-06 00:55:58
850   2025-08-10 17:35:29
710   2026-01-03 18:02:34
626   2026-03-17 14:51:45
457   2026-03-10 15:40:29

Date dtype: datetime64[ns]


In [26]:
# B) Data Quality Audit

print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

# --- Problem 2: Duplicate Reviews ---
print("\nProblem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

# --- Problem 3: Date Format ---
print("\nProblem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 365
  Duplicate review IDs    : 0
  Empty review texts      : 0

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[ns]
  Sample values: 2026-05-15 12:22:49
  Target format: YYYY-MM-DD (string or date object)


In [33]:
# C) Cleaning Strategy

# Step 0:- Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

# Step 1:- Handle Missing Values

before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"\nRemoved {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

# Step 2:- Remove Duplicates

before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"\nRemoved {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

# Step 3:- Normalize Dates

print("\nBefore normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

# Step 4:- Clean Review Text

def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"\nBefore: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

# Step 5:- Validate Ratings

# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"\nInvalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Starting with: 1500 reviews

Removed 0 rows with missing critical data
Remaining: 1500 reviews

Removed 0 duplicate reviews
Remaining: 1500 reviews

Before normalization:
0   2026-05-15 12:22:49
1   2026-05-15 12:07:21
2   2026-05-14 18:52:51
dtype: datetime64[ns]

After normalization:
0    2026-05-15
1    2026-05-15
2    2026-05-14
dtype: object

Date range: 2025-02-14 to 2026-05-15

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning

Invalid ratings (outside 1–5): 0
Remaining: 1500 reviews
Rating dtype: int64


In [36]:
# Step 6:- Final output

# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (1500, 5)


,review,rating,date,bank,source
0,It's not allowing me to transfer money.,2,2026-05-15,Commercial Bank of Ethiopia,Google Play
1,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15,Commercial Bank of Ethiopia,Google Play
2,good,5,2026-05-14,Dashen Bank,Google Play
3,Very Annoying App i tried to open virtual bank...,1,2026-05-14,Dashen Bank,Google Play
4,good app but it was doesnt work other bank tra...,5,2026-05-14,Dashen Bank,Google Play
5,good,5,2026-05-14,Bank of Abyssinia,Google Play
6,good,5,2026-05-14,Dashen Bank,Google Play
7,"i swear to god , By using this app, I won a Sa...",5,2026-05-14,Dashen Bank,Google Play
8,good and easier to used,5,2026-05-14,Dashen Bank,Google Play
9,good,5,2026-05-14,Dashen Bank,Google Play


In [38]:
# Save to CSV
import os

target_directory = r'C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed'
os.makedirs(target_directory, exist_ok=True)

output_path = os.path.join(target_directory, 'combined_bank_reviews_raw.csv')
df_clean.to_csv(output_path, index=False)

print(f"✅ Master dataset successfully saved to:\n{output_path}")

✅ Master dataset successfully saved to:
C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed\combined_bank_reviews_raw.csv


In [ ]:
# Step 7: Preprocessing Report

print("=" * 55)
print("  PREPROCESSING REPORT — CBE, BOA, Dashen Banks Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

In [ ]:
import pandas as pd

# List of banks to process based on your combined dataset
banks_to_report = ["Commercial Bank of Ethiopia", "Bank of Abyssinia", "Dashen Bank"]

for bank in banks_to_report:
    # Filter dataframes for the current bank
    df_raw_bank = df_raw[df_raw['bank'] == bank]
    df_clean_bank = df_clean[df_clean['bank'] == bank]
    
    # Skip if no data exists for this bank to prevent division-by-zero errors
    if len(df_raw_bank) == 0:
        print(f"\n⚠️ No data found for {bank}. Skipping...")
        continue

    print("=" * 55)
    print(f"   PREPROCESSING REPORT — {bank}")
    print("=" * 55)

    original_count = len(df_raw_bank)
    final_count    = len(df_clean_bank)
    removed_total  = original_count - final_count
    retention_rate = (final_count / original_count * 100)

    print(f"\n  Raw reviews collected  : {original_count:>6}")
    print(f"  Reviews after cleaning : {final_count:>6}")
    print(f"  Reviews removed        : {removed_total:>6}")
    print(f"  Data retention rate    : {retention_rate:>5.1f}%")

    quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
    print(f"  Data quality           : {quality}")

    # Handle missing dates gracefully if all records were cleared
    if final_count > 0:
        print(f"\n  Date range : {df_clean_bank['date'].min()}  to  {df_clean_bank['date'].max()}")

        print("\n  Rating distribution:")
        for rating in sorted(df_clean_bank['rating'].unique(), reverse=True):
            count = (df_clean_bank['rating'] == rating).sum()
            pct   = count / final_count * 100
            bar   = '█' * (count // 5)  # Visual scale helper
            print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

        print("\n  Text length stats:")
        lengths = df_clean_bank['review'].str.len()
        print(f"    Min    : {lengths.min()} characters")
        print(f"    Median : {lengths.median():.0f} characters")
        print(f"    Max    : {lengths.max()} characters")
    else:
        print("\n  ⚠️ No records remaining after data cleaning step.")

    print("\n  Columns in final CSV:")
    for col in df_clean.columns:
        print(f"    - {col}")

    print("\n" + "=" * 55 + "\n")